# AG — Posicionamento de CFUs na malha

Versão autocontida do projeto (`directed_graph.py` + `ag.py`) para rodar no Google Colab.

**Como usar:** suba este arquivo no Colab (*File → Upload notebook*) e execute as células em ordem. Matplotlib já vem instalado no Colab e o gráfico aparece inline.

## 1. Funções do grafo dirigido (`directed_graph.py`)

Ordenação topológica (Kahn), custo de aresta (Manhattan a partir da última célula da cadeia + delay interno), caminho crítico, bounding box, colisões e caminhos source→sink.

In [ ]:
import random
from collections import deque

def get_topologic_order(vertices, graph):

    # Grau de entrada (in-degree): número de arestas que CHEGAM no vértice
    in_degree = {vertex: 0 for vertex in vertices}
    for adjacent_list in graph.values():
        for adjacent_vertex in adjacent_list:
            in_degree[adjacent_vertex] += 1

    queue = deque([vertex for vertex in vertices if in_degree[vertex] == 0])
    topologic_order = []

    while queue:
        first_out_vertex = queue.popleft()
        topologic_order.append(first_out_vertex)

        for vertex in graph[first_out_vertex]:
            in_degree[vertex] -= 1
            if in_degree[vertex] == 0:
                queue.append(vertex)

    if len(topologic_order) != len(vertices):
        raise ValueError("graph contains a cycle!")

    return topologic_order


def get_large_constant(graph, grid_width, grid_height):
    number_of_nets = sum(1 for destinations in graph.values() if destinations)
    return number_of_nets * (grid_width + grid_height) + 1


def get_occupied_cells(individual, vertices, cfu_count):
    occupied_cells = []
    for vertex, (column, row) in zip(vertices, individual):
        width = max(1, cfu_count[vertex])
        for offset in range(width):
            occupied_cells.append((column + offset, row))
    return occupied_cells


def has_shared_position(individual, vertices, cfu_count):
    occupied_cells = get_occupied_cells(individual, vertices, cfu_count)
    return len(set(occupied_cells)) != len(occupied_cells)


def get_edge_cost(source, destination, positions, cfu_count):
    source_column, source_row = positions[source]
    destination_column, destination_row = positions[destination]

    # a saída de uma cadeia de CFUs é a última célula; a entrada do destino é a primeira
    source_exit_column = source_column + max(1, cfu_count[source]) - 1

    # manhattan distance
    wire_length = abs(source_exit_column - destination_column) + abs(source_row - destination_row)
    return wire_length + cfu_count[source]


def get_critical_path_cost(positions, graph, cfu_count, topologic_order):
    longest_distance = {vertex: 0 for vertex in topologic_order}

    for source in topologic_order:
        for destination in graph[source]:
            candidate = longest_distance[source] + get_edge_cost(source, destination, positions, cfu_count)
            if candidate > longest_distance[destination]:
                longest_distance[destination] = candidate

    # (graph[vertex] == []) é o mesmo que (not graph[vertex]), verifica se o vertice é output.
    return max(longest_distance[vertex] for vertex in topologic_order if not graph[vertex])


def get_total_bounding_box_cost(positions, graph):
    total_bounding_box_cost = 0
    for source, destinations in graph.items():
        if not destinations:
            continue

        net = [source] + destinations

        columns = [positions[vertex][0] for vertex in net]
        rows = [positions[vertex][1] for vertex in net]

        total_bounding_box_cost += (max(columns) - min(columns)) + (max(rows) - min(rows))
    return total_bounding_box_cost


def gera_dag(seed, n_in, n_out, n_camadas, tam_camada, malha_largura, malha_altura, max_fan_in=4):
    """Gera um DAG em camadas: toda aresta sai de uma camada e chega numa
    camada mais à frente, logo ciclos são impossíveis. Cada vértice recebe
    de 1 a max_fan_in pais de camadas anteriores.
    Retorna (vertices, graph, cfu_count, vertices_fixos)."""
    rng = random.Random(seed)

    layers = [[f'IN{i}' for i in range(n_in)]]
    for c in range(n_camadas):
        layers.append([f'U{c}_{j}' for j in range(tam_camada)])
    layers.append([f'OUT{i}' for i in range(n_out)])

    vertices = [v for camada in layers for v in camada]
    graph = {v: [] for v in vertices}
    fan_in = {v: 0 for v in vertices}

    # passo 1: sorteia os pais (entradas) de cada vértice fora da camada 0
    for li in range(1, len(layers)):
        for v in layers[li]:
            k = rng.randint(1, max_fan_in)
            pais = set()
            while len(pais) < k:
                # 70% dos pais vêm da camada imediatamente anterior;
                # 30% de qualquer camada anterior (aresta que "pula" camada)
                if rng.random() < 0.7:
                    camada_pai = layers[li - 1]
                else:
                    camada_pai = rng.choice(layers[:li])
                pais.add(rng.choice(camada_pai))
            for p in pais:
                graph[p].append(v)
            fan_in[v] = len(pais)

    # passo 2: garante >= 1 filho para todo vértice fora da última camada,
    # senão ele viraria uma saída extra (subgrafo desconexo)
    for li in range(len(layers) - 1):
        for v in layers[li]:
            if not graph[v]:
                for lj in range(li + 1, len(layers)):
                    candidatos = [u for u in layers[lj] if fan_in[u] < max_fan_in]
                    if candidatos:
                        alvo = rng.choice(candidatos)
                        graph[v].append(alvo)
                        fan_in[alvo] += 1
                        break
                else:
                    raise RuntimeError('sem capacidade de fan-in nas camadas à frente')

    # I/O não tem CFU; vértices internos têm cadeias de 1 a 10 células
    cfu_count = {}
    for li, camada in enumerate(layers):
        for v in camada:
            cfu_count[v] = 0 if li == 0 or li == len(layers) - 1 else rng.randint(1, 10)

    # entradas espalhadas no topo, saídas na base (fixas)
    vertices_fixos = {}
    for i, v in enumerate(layers[0]):
        vertices_fixos[v] = (int((i + 0.5) * malha_largura / n_in), 0)
    for i, v in enumerate(layers[-1]):
        vertices_fixos[v] = (int((i + 0.5) * malha_largura / n_out), malha_altura - 1)

    return vertices, graph, cfu_count, vertices_fixos


def problema_complexo(seed=7, malha_largura=100, malha_altura=100,
                      n_in=8, n_out=6, n_camadas=12, tam_camada=10, max_fan_in=4):
    """Problema grande: malha 100x100 com DAG gerada em camadas
    (134 vértices por padrão: 8 entradas + 12x10 internos + 6 saídas,
    fan-in até max_fan_in). Seed fixa -> grafo reproduzível."""
    vertices, graph, cfu_count, vertices_fixos = gera_dag(
        seed, n_in, n_out, n_camadas, tam_camada,
        malha_largura, malha_altura, max_fan_in)
    return {
        'malha_largura': malha_largura,
        'malha_altura': malha_altura,
        'vertices': vertices,
        'graph': graph,
        'cfu_count': cfu_count,
        'vertices_fixos': vertices_fixos
    }

## 2. Configuração do problema e operadores do AG (`ag.py`)

Cromossomo = lista de posições `(x, y)` na ordem fixa de `VERTICES`. Vértices fixos (`VERTICES_FIXOS`) não se movem. `CFU_COUNT` = largura em células (delay interno).

O problema simples (malha 5×5, 4 vértices) está direto no código. Com `COMPLEXO = True`, a configuração é substituída pela de `problema_complexo()` (malha 100×100, ~134 vértices em camadas, fan-in até 4 — seed fixa, reproduzível).

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

from typing import Final

COMPLEXO = True

MALHA_LARGURA = 5
MALHA_ALTURA = 5

VERTICES = ['A', 'B', 'C', 'D']
GRAPH = {
    'A': ['B', 'C'],
    'B': ['C', 'D'],
    'C': ['D'],
    'D': []
}
CFU_COUNT = {'A': 0, 'B': 1, 'C': 3, 'D': 0}
VERTICES_FIXOS = {'A': (2, 0), 'D': (2, MALHA_ALTURA - 1)}

VERBOSE: Final[bool] = not COMPLEXO

# A posição do vértice será uma tupla (x,y) e cada individuo tem a mesma quantidade de posições de vértices
TAM_CROMO: Final[int] = len(VERTICES)
TAM_POP: Final[int] = TAM_CROMO * 2 if not COMPLEXO else 150
N_GERACOES: Final[int] = 30 if not COMPLEXO else 2000
GERACOES_PARADA: Final[int] = 15 if not COMPLEXO else 100
TAXA_MUTACAO: Final[float] = 0.5


pop = [[(0, 0)] * TAM_CROMO for _ in range(TAM_POP)]
nova_pop = [[(0, 0)] * TAM_CROMO for _ in range(TAM_POP)]
pais = [[(0, 0)] * TAM_CROMO for _ in range(2)]
filhos = [[(0, 0)] * TAM_CROMO for _ in range(2)]
nota_pop = np.zeros((TAM_POP, 3))


def init_pop():
    global pop
    pop = [individuo_aleatorio() for _ in range(TAM_POP)]


def avalia_pop():
    global nota_pop
    soma_apt = 0

    topologic_order = get_topologic_order(VERTICES, GRAPH)
    large_constant = get_large_constant(GRAPH, MALHA_LARGURA, MALHA_ALTURA)

    for index_individuo in range(TAM_POP):
        apt = 0
        individuo = pop[index_individuo]

        # penalidade
        if has_shared_position(individuo, VERTICES, CFU_COUNT):
            apt = float('inf')
        else:
            positions = dict(zip(VERTICES, individuo))

            critical_path_cost = get_critical_path_cost(positions, GRAPH, CFU_COUNT, topologic_order)
            total_bounding_box_cost = get_total_bounding_box_cost(positions, GRAPH)

            total_cost = critical_path_cost * large_constant + total_bounding_box_cost
            apt = total_cost

        nota_pop[index_individuo][0] = index_individuo
        nota_pop[index_individuo][1] = apt
        nota_pop[index_individuo][2] = -1

        # descarta apts que sofreram penalidade
        if apt < float('inf'):
            soma_apt = soma_apt + apt

    minsum = 0
    nota_pop = nota_pop[nota_pop[:, 1].argsort()]

    for index_individuo in range(TAM_POP):
        if (soma_apt > 0) and (nota_pop[index_individuo][1] > 0):
            nota_pop[index_individuo][2] = ((1 / nota_pop[index_individuo][1]) / soma_apt) * 100
        else:
            nota_pop[index_individuo][2] = 0
        minsum = minsum + nota_pop[index_individuo][2]

    for i in range(TAM_POP):
        if(minsum > 0):
            nota_pop[i][2] = nota_pop[i][2] / minsum
        else:
            nota_pop[i][2] = 0


def seleciona_pais():
    global pais
    r_pai1 = random.random()
    r_pai2 = random.random()

    acum = 0
    for i in range(TAM_POP):
        acum = acum + nota_pop[i][2]
        if acum >= r_pai1:
            pais[0] = list(pop[int(nota_pop[i][0])])
            break
    else:
        pais[0] = list(pop[int(nota_pop[0][0])])

    acum = 0
    for i in range(TAM_POP):
        acum = acum + nota_pop[i][2]
        if acum >= r_pai2:
            pais[1] = list(pop[int(nota_pop[i][0])])
            break
    else:
        pais[1] = list(pop[int(nota_pop[0][0])])


def cruza_pais():
    global filhos

    r_cruza = random.random()
    if(r_cruza < 0.80):
        corte = random.randint(0, TAM_CROMO - 1)
        filhos[0][:corte] = pais[0][:corte]
        filhos[0][corte:] = pais[1][corte:]
        filhos[1][corte:] = pais[0][corte:]
        filhos[1][:corte] = pais[1][:corte]
    else:
        filhos[0] = pais[0]
        filhos[1] = pais[1]


def muta_filhos():
    global filhos

    for i in range(TAM_CROMO):
        r_muta = random.random()
        if(r_muta < TAXA_MUTACAO):
            filhos[0][i] = posicao_aleatoria(VERTICES[i])

    for i in range(TAM_CROMO):
        r_muta = random.random()
        if(r_muta < TAXA_MUTACAO):
            filhos[1][i] = posicao_aleatoria(VERTICES[i])


def elitismo(qtde):
    global nova_pop
    for i in range(qtde):
        nova_pop[i] = pop[int(nota_pop[i][0])].copy()


def imprime_pop():
    for index, individuo in enumerate(pop):
        print(f"individuo ({index}): {individuo}")


def imprime_nota_pop():
    acum = 0
    for i in range(TAM_POP):
        acum = acum + nota_pop[i][2]
        print('Individuo: ', nota_pop[i][0], '- Nota: ', nota_pop[i][1], '- (%): ', nota_pop[i][2], ' - Acum: ', acum)


def imprime_melhor():
    melhor = pop[int(nota_pop[0][0])]
    positions = dict(zip(VERTICES, melhor))
    topologic_order = get_topologic_order(VERTICES, GRAPH)
    caminho_critico = get_critical_path_cost(positions, GRAPH, CFU_COUNT, topologic_order)
    bbox_total = get_total_bounding_box_cost(positions, GRAPH)
    print(f'>> Melhor: {melhor} - Nota: {nota_pop[0][1]}'
          f' (caminho crítico: {caminho_critico}, bounding box: {bbox_total})'
          f' - (%): {nota_pop[0][2]}')


def posicao_aleatoria(vertex):
    if vertex in VERTICES_FIXOS:
        return VERTICES_FIXOS[vertex]

    width = max(1, CFU_COUNT[vertex])
    x = random.randint(0, MALHA_LARGURA - width)
    y = random.randint(1, MALHA_ALTURA - 2)
    return (x, y)


def individuo_aleatorio():
    while True:
        individual = [posicao_aleatoria(vertex) for vertex in VERTICES]
        if not has_shared_position(individual, VERTICES, CFU_COUNT):
            return individual


def plota_melhor(individuo):
    positions = dict(zip(VERTICES, individuo))
    malha_grande = max(MALHA_LARGURA, MALHA_ALTURA) > 50
    fig, ax = plt.subplots(figsize=(10, 10))

    # células ocupadas (vértice ocupa max(1, cfu_count) colunas)
    for vertex, (x, y) in positions.items():
        width = max(1, CFU_COUNT[vertex])
        ax.add_patch(plt.Rectangle((x, y), width, 1, alpha=0.4))
        ax.text(x + width / 2, y + 0.5, vertex, ha='center', va='center',
                fontsize=4 if malha_grande else 10)

    cmap = plt.get_cmap('tab10')
    total_bbox = 0
    net_index = 0

    for source, destinations in GRAPH.items():
        if not destinations:
            continue

        cor = cmap(net_index % 10)
        net_index += 1

        net = [source] + destinations
        min_x = min(positions[v][0] for v in net)
        max_x = max(positions[v][0] for v in net)
        min_y = min(positions[v][1] for v in net)
        max_y = max(positions[v][1] for v in net)

        bbox_cost = (max_x - min_x) + (max_y - min_y)
        total_bbox += bbox_cost

        if not malha_grande:
            pad = net_index * 0.07
            ax.add_patch(plt.Rectangle((min_x + pad, min_y + pad),
                                       max_x - min_x + 1 - 2 * pad,
                                       max_y - min_y + 1 - 2 * pad,
                                       fill=False, ls='--', edgecolor=cor,
                                       lw=1.5))
            ax.plot([], [], color=cor, ls='--', lw=1.5,
                    label=f'{source}→{{{",".join(destinations)}}}: {bbox_cost}')

        for dest in destinations:
            sx, sy = positions[source]
            dx, dy = positions[dest]
            ax.annotate('', xy=(dx + 0.5, dy + 0.5), xytext=(sx + 0.5, sy + 0.5),
                        arrowprops=dict(arrowstyle='->', color=cor,
                                        lw=0.4 if malha_grande else 1.5,
                                        alpha=0.5 if malha_grande else 1.0))

    topologic_order = get_topologic_order(VERTICES, GRAPH)
    large_constant = get_large_constant(GRAPH, MALHA_LARGURA, MALHA_ALTURA)
    caminho_critico = get_critical_path_cost(positions, GRAPH, CFU_COUNT, topologic_order)
    fitness = caminho_critico * large_constant + total_bbox

    ax.set_title(f'fitness: {fitness:.0f} = caminho crítico {caminho_critico}'
                 f' × {large_constant:.0f} + bounding box {total_bbox}')
    if not malha_grande:
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9,
                  title='bounding box por net')
    ax.set_xlim(0, MALHA_LARGURA)
    ax.set_ylim(0, MALHA_ALTURA)
    ax.invert_yaxis()

    ax.set_xticks(range(MALHA_LARGURA + 1))
    ax.set_yticks(range(MALHA_ALTURA + 1))

    if malha_grande:
        ax.tick_params(axis='x', labelsize=3, labelrotation=90)
        ax.tick_params(axis='y', labelsize=3)

    ax.grid(True)
    plt.show()  # use plt.savefig('melhor.png') para salvar

## 3. Loop principal do AG

In [ ]:
if COMPLEXO:
    problema = problema_complexo()
    MALHA_LARGURA = problema['malha_largura']
    MALHA_ALTURA = problema['malha_altura']
    VERTICES = problema['vertices']
    GRAPH = problema['graph']
    CFU_COUNT = problema['cfu_count']
    VERTICES_FIXOS = problema['vertices_fixos']
    TAM_CROMO = len(VERTICES)
    pop = [[(0, 0)] * TAM_CROMO for _ in range(TAM_POP)]
    nova_pop = [[(0, 0)] * TAM_CROMO for _ in range(TAM_POP)]
    pais = [[(0, 0)] * TAM_CROMO for _ in range(2)]
    filhos = [[(0, 0)] * TAM_CROMO for _ in range(2)]
    nota_pop = np.zeros((TAM_POP, 3))

init_pop()
if VERBOSE:
    imprime_pop()

melhor_fitness = float('inf')
geracoes_estagnadas = 0
for i in range(N_GERACOES):
    print('GERAÇÃO: ', i)
    avalia_pop()

    if VERBOSE:
        imprime_nota_pop()
    else:
        print('  melhor fitness:', nota_pop[0][1])

    # criterio de parada
    if (nota_pop[0][1] < melhor_fitness):
        melhor_fitness = nota_pop[0][1]
        geracoes_estagnadas = 0
    else:
        geracoes_estagnadas += 1
        if geracoes_estagnadas == GERACOES_PARADA:
            break

    # preserva n melhores
    n_melhores = 2
    elitismo(n_melhores)

    while(n_melhores < TAM_POP):
        seleciona_pais()
        cruza_pais()
        muta_filhos()

        nova_pop[n_melhores] = filhos[0].copy()
        nova_pop[n_melhores+1] = filhos[1].copy()
        n_melhores = n_melhores + 2

    pop = nova_pop.copy()
    nova_pop = [[(0, 0)] * TAM_CROMO for _ in range(TAM_POP)]

imprime_melhor()
plota_melhor(pop[int(nota_pop[0][0])])